# 🪵 TimberGrade - Model Training (Google Colab)
**AI-Based Wooden Plank Defect Detection and Quality Grading System**

Ye notebook YOLOv8 model ko lakdi ke defects pehchanna sikhata hai.

### Kaise chalana hai
1. Upar menu mein **Runtime → Change runtime type → T4 GPU** chuno → Save
2. Har cell ko **upar se neeche, ek-ek karke** chalao (cell par ▶️ button)
3. Last cell `best.pt` file download karega → use TimberGrade folder mein daalna hai

⏱️ Poora training ~1 se 1.5 ghanta lega. Beech mein tab band mat karna.

## Step 1: GPU check karo
Neeche **Tesla T4** jaisa kuch dikhna chahiye. Agar error aaye toh Step 0 (GPU select) dobara karo.

In [ ]:
!nvidia-smi

## Step 2: Libraries install karo

In [ ]:
!pip install -q ultralytics kagglehub

## Step 3: Dataset download karo
Kaggle se **Large Scale Image Dataset of Wood Surface Defects** (4,000 photos, YOLO labels ke saath).

Agar yahan login / 403 error aaye:
1. kaggle.com par login karo → Profile → **Settings → API → Create New Token** (`kaggle.json` download hoga)
2. Neeche wale cell ke `USE_KAGGLE_JSON = True` kar do aur cell dobara chalao, wo file upload karne ko bolega

In [ ]:
import os, kagglehub
USE_KAGGLE_JSON = False

if USE_KAGGLE_JSON:
    from google.colab import files
    up = files.upload()  # kaggle.json chuno
    os.makedirs("/root/.kaggle", exist_ok=True)
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(list(up.values())[0])
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

DATA_DIR = kagglehub.dataset_download("nomihsa965/large-scale-image-dataset-of-wood-surface-defects")
print("Dataset yahan hai:", DATA_DIR)
!du -sh "$DATA_DIR"
!find "$DATA_DIR" -maxdepth 3 -type d | head -30

## Step 4: Dataset ko YOLO ke liye taiyaar karo
Ye cell apne aap photos aur unke label files dhoondhkar jodta hai, aur 85% training / 15% testing mein baant deta hai.

In [ ]:
import os, glob, random, shutil, re, yaml
from collections import Counter

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
imgs, labels = {}, {}
for root, _, fs in os.walk(DATA_DIR):
    for f in fs:
        stem, ext = os.path.splitext(f)
        p = os.path.join(root, f)
        if ext.lower() in IMG_EXT:
            imgs[stem] = p
        elif ext.lower() == ".txt" and stem.lower() not in {"classes", "readme", "obj", "labels"}:
            labels[stem] = p
print(f"Photos mili: {len(imgs)}   Label files mili: {len(labels)}")

# ---- Class names dhoondhna (yaml / classes.txt) ----
names = None
for root, _, fs in os.walk(DATA_DIR):
    for f in fs:
        p = os.path.join(root, f)
        if f.endswith((".yaml", ".yml")):
            try:
                y = yaml.safe_load(open(p))
                if isinstance(y, dict) and "names" in y:
                    n = y["names"]; names = [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
            except Exception: pass
        elif f.lower() in {"classes.txt", "obj.names"}:
            names = [l.strip() for l in open(p) if l.strip()]
if not names:
    names = ["Quartzity", "Live_Knot", "Marrow", "resin", "Dead_Knot", "knot_with_crack", "Knot_missing", "Crack"]
    print("⚠️ Class list file nahi mili, default list use kar rahe hain")

# ---- Photo + label jodna aur check karna ----
pairs, cls_count, bad = [], Counter(), 0
for stem, ip in imgs.items():
    lp = labels.get(stem)
    if not lp: continue
    ok, rows = True, []
    for line in open(lp):
        v = line.split()
        if not v: continue
        if len(v) != 5: ok = False; break
        c, *xywh = v
        xywh = list(map(float, xywh))
        if any(x < 0 or x > 1.0001 for x in xywh): ok = False; break
        rows.append(int(float(c)))
    if not ok: bad += 1; continue
    cls_count.update(rows); pairs.append((ip, lp))

max_id = max(cls_count) if cls_count else -1
while len(names) <= max_id: names.append(f"defect_{len(names)}")
print(f"Jode gaye pairs: {len(pairs)}   Kharab labels (chhode): {bad}")
print("Classes:", names)
for k, v in sorted(cls_count.items()): print(f"  {k} {names[k]:<16} {v} boxes")

# ---- Train / Val split ----
OUT = "/content/wood"
shutil.rmtree(OUT, ignore_errors=True)
random.seed(42); random.shuffle(pairs)
n_val = int(len(pairs) * 0.15)
for split, items in [("val", pairs[:n_val]), ("train", pairs[n_val:])]:
    os.makedirs(f"{OUT}/images/{split}"); os.makedirs(f"{OUT}/labels/{split}")
    for ip, lp in items:
        stem = os.path.splitext(os.path.basename(ip))[0]
        os.symlink(ip, f"{OUT}/images/{split}/{os.path.basename(ip)}")
        shutil.copy(lp, f"{OUT}/labels/{split}/{stem}.txt")
yaml.safe_dump({"path": OUT, "train": "images/train", "val": "images/val", "names": dict(enumerate(names))},
               open(f"{OUT}/data.yaml", "w"))
print(f"\nTrain: {len(pairs) - n_val}   Val: {n_val}   ✅ data.yaml ready")

## Step 5: Ek sample photo dekho (labels sahi hain ya nahi)

In [ ]:
import cv2, matplotlib.pyplot as plt
ip = glob.glob(f"{OUT}/images/train/*")[0]
stem = os.path.splitext(os.path.basename(ip))[0]
im = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
for line in open(f"{OUT}/labels/train/{stem}.txt"):
    c, x, y, w, h = map(float, line.split())
    x1, y1, x2, y2 = int((x - w/2)*W), int((y - h/2)*H), int((x + w/2)*W), int((y + h/2)*H)
    cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 6)
    cv2.putText(im, names[int(c)], (x1, max(30, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 0, 0), 4)
plt.figure(figsize=(16, 6)); plt.imshow(im); plt.axis("off"); plt.show()

## Step 6: Model train karo 🚀
Ye sabse lamba step hai (~1 ghanta). Har epoch ke baad `mAP50` badhna chahiye.

- `epochs=30` → model 30 baar poora data dekhega
- Time kam hai toh `epochs=15` kar sakte ho (accuracy thodi kam hogi)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{OUT}/data.yaml",
    epochs=30,
    imgsz=1024,      # lakdi ki photos lambi hain, isliye bada size
    batch=8,
    patience=10,     # 10 epoch tak sudhaar na ho toh ruk jaayega
    project="/content/runs",
    name="timbergrade",
    exist_ok=True,
)

## Step 7: Results dekho (report ke liye screenshots yahin se lo)

In [ ]:
from IPython.display import Image, display
RUN = "/content/runs/timbergrade"
metrics = YOLO(f"{RUN}/weights/best.pt").val(data=f"{OUT}/data.yaml", imgsz=1024, batch=8)
print(f"\n📊 mAP50     : {metrics.box.map50*100:.1f}%")
print(f"📊 mAP50-95  : {metrics.box.map*100:.1f}%")
print(f"📊 Precision : {metrics.box.mp*100:.1f}%")
print(f"📊 Recall    : {metrics.box.mr*100:.1f}%")
for f in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    p = f"{RUN}/{f}"
    if os.path.exists(p): print("\n", f); display(Image(p, width=1000))

## Step 8: Nayi photo par test karo

In [ ]:
best = YOLO(f"{RUN}/weights/best.pt")
for ip in glob.glob(f"{OUT}/images/val/*")[:3]:
    r = best(ip, imgsz=1024, conf=0.25)[0]
    plt.figure(figsize=(16, 6)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off"); plt.show()
    print([f"{r.names[int(c)]} {float(s)*100:.0f}%" for c, s in zip(r.boxes.cls, r.boxes.conf)])

## Step 9: Model + results download karo ✅
Do files download hongi:
- **best.pt** → TimberGrade folder ke andar `models` folder mein daalna
- **training_results.zip** → report ke graphs aur photos

In [ ]:
from google.colab import files
shutil.make_archive("/content/training_results", "zip", RUN)
files.download(f"{RUN}/weights/best.pt")
files.download("/content/training_results.zip")